# notebook 10 — 本丸 (1)：作用の実効相関と平均場の相転移

引き継ぎ書 open 1（本丸）の検証。これまで（nb05〜09）で確定したのは「**素の二点相関 `cos2θ`
を直接いじっても非コンパクト/単一時間は出ない**」こと。共通根因は `cos2θ` の周期性・有界性・
ランク2。残る唯一の原理的余地は、作用 `S=−βΣe^{−d}` が定める**実効二点相関**が、素の `cos2θ`
と違って臨界点をまたいで構造を変えうる、という統計力学的機構（位相転移）であった。

この notebook の問いは二本立て（前 Chat で確定した設計）：

- **判定I（非コンパクト化）**：相関長 `ξ` が発散する臨界点 `β*` が存在するか。
- **判定II（ローレンツ化＝符号）**：もし `β*` が存在するなら、その臨界点で発散するモードが
  **非周期モード**に乗り換わり、`sign(cos2θ)` の π/4 ごとの周期反転が解消するか。

判定I が出ても判定II が自動でついてこないことは A-7（非コンパクト化とローレンツ化は別操作）で
確認済み。両方出て初めて open 1 と open 4 が同時に閉じる。

**規律（教訓D）**：綺麗さでなく原理への忠実さで判断する。作った量が与件 `cos2θ` から導かれるか
を必ず確認する。成功・失敗どちらの早すぎる一般化も避ける。

## 10.1 モデルの定義（与件から導く）

与件：`E(θ)=cos2θ/(2N)`、`N=3`。距離 `d=−log|C|`、作用 `S=−βΣ_{<ij>} e^{−d_{ij}}`。
結合の重み `e^{−d} = |C| = |cos2(θ_i−θ_j)|/(2N)`。

**自由度**：各サイト `i` に角度設定 `s_i∈[0,2π)`（連続スピン）。
**結合**：相関行列の台＝**全結合**（全ペアを結ぶ）。これは新しい格子を手で入れていない——
相関行列はもともと全ペアで定義済みであり、全結合は与件が持つ構造そのもの（教訓D-3 を回避）。

平均場が解析的に解けるのは、`cos2(s−s') = cos2s·cos2s' + sin2s·sin2s'` が **rank-2** だから
（nb05 の代数的事実）。秩序変数は `m=(⟨cos2s⟩, ⟨sin2s⟩)`、その大きさ `R=|m|` が秩序パラメータ。
角度並進 `s→s+δ` は `(cos2s,sin2s)` を回す O(2) 対称性。`R=0`＝対称（円のまま）、
`R>0`＝対称性の破れ（方向選択）。

**符号の扱い**：結合に相関の**符号つき** `cos2`（絶対値でなく）を用いる。前 Chat で確認した通り、
絶対値核は Jentzsch–Perron で常にギャップ正＝相関長有限となり臨界点が構造的に出ない。符号を
残すことが判定II（open 4）に触れる唯一の道。

In [1]:
import numpy as np
from numpy.linalg import eigh

N = 3  # 与件（QCDとの同定）

def self_consistent_R(beta, grid=4000, iters=6000, seed=0.3, seed_angle=0.0):
    # 全結合・符号つき cos2 結合の平均場 自己無撞着解。R=|m| を返す。
    s = np.linspace(0, 2*np.pi, grid, endpoint=False)
    c2, s2 = np.cos(2*s), np.sin(2*s)
    mx, my = seed*np.cos(seed_angle), seed*np.sin(seed_angle)
    for _ in range(iters):
        f = (beta/(2*N))*(mx*c2 + my*s2); f -= f.max()
        w = np.exp(f); w /= w.sum()
        nx, ny = (w*c2).sum(), (w*s2).sum()
        if abs(nx-mx)+abs(ny-my) < 1e-14:
            mx, my = nx, ny; break
        mx, my = nx, ny
    return np.hypot(mx, my)

print("平均場 秩序パラメータ R(beta)：")
print("beta :   R")
for b in [0.5,2,5,8,10,11,11.5,12,12.5,13,14,16,20,30]:
    print(f"{b:5.1f}: {self_consistent_R(b):.6f}")

平均場 秩序パラメータ R(beta)：
beta :   R
  0.5: 0.000000
  2.0: 0.000000
  5.0: 0.000000
  8.0: 0.000000
 10.0: 0.000000
 11.0: 0.000000
 11.5: 0.000000


 12.0: 0.012897
 12.5: 0.279059
 13.0: 0.382114
 14.0: 0.508799
 16.0: 0.647237
 20.0: 0.773130
 30.0: 0.876823


## 10.2 判定I：臨界点 `β*` の同定と検証

`R` が立ち上がる点を二分法で求め、(a) シード非依存（初期値の人工物でない）、
(b) `β*` 以下で微小シードが 0 に減衰する（真の無秩序相）、を確認する。
平均場の線形化から臨界条件 `1 = (β/2N)·⟨cos²2s⟩₀ = (β/2N)·(1/2)`、すなわち **`β*=4N`**。

In [2]:
# (a) シード非依存性
print("シード非依存性 @ beta=20（初期値の大きさ・角度を変える）：")
for sd in [1e-6, 0.01, 0.9]:
    vals = [self_consistent_R(20, seed=sd, seed_angle=a) for a in [0.0,1.0,2.5]]
    print(f"  seed={sd:.0e}: " + "  ".join(f"R={v:.6f}" for v in vals))

# (b) 微小シードの減衰／成長
print("\n微小シード(1e-6) test：beta* 以下で R→0、以上で成長：")
for b in [6,9,11,11.5,12,12.5,13,14,16]:
    print(f"  beta={b:5.1f}: R={self_consistent_R(b, seed=1e-6):.6e}")

# (c) 二分法で beta*
def R_small(b): return self_consistent_R(b, seed=1e-5, iters=8000)
lo, hi = 10.0, 14.0
for _ in range(40):
    mid = 0.5*(lo+hi)
    if R_small(mid) > 1e-4: hi = mid
    else: lo = mid
print(f"\nbeta* (二分法) ~ {0.5*(lo+hi):.5f}")
print(f"解析的予言 beta* = 4N = {4*N}")

シード非依存性 @ beta=20（初期値の大きさ・角度を変える）：
  seed=1e-06: R=0.773130  R=0.773130  R=0.773130
  seed=1e-02: R=0.773130  R=0.773130  R=0.773130
  seed=9e-01: R=0.773130  R=0.773130  R=0.773130

微小シード(1e-6) test：beta* 以下で R→0、以上で成長：
  beta=  6.0: R=7.494005e-15
  beta=  9.0: R=2.393920e-14
  beta= 11.0: R=1.024736e-13
  beta= 11.5: R=2.222111e-13
  beta= 12.0: R=1.000000e-06
  beta= 12.5: R=2.790592e-01
  beta= 13.0: R=3.821139e-01
  beta= 14.0: R=5.087991e-01
  beta= 16.0: R=6.472375e-01



beta* (二分法) ~ 12.00345
解析的予言 beta* = 4N = 12


**判定I の結果**：臨界点 `β*≈12.00` が存在し、解析的予言 `β*=4N` と一致。シード非依存・
`β*`以下で無秩序相、で本物の連続相転移と確認。臨界点が与件の `N`（QCD同定値 3）から `β*=4N`
として**出力**されている点が重要（手で入れた量でない）。

→ **判定I はクリア**：相関長発散の臨界点は存在する。非コンパクト化の入口に立った。

## 10.3 判定II：破れた相での実効相関の構造（符号の問題）

本丸の本体。破れた相 `β>β*` で実効相関行列 `C_eff(θ,θ')` を組み、その**フーリエ次数**と
**ランク**を見る。問い：支配モードが依然 k=2/4 の周期モードのままか、非周期モードに乗り換わるか。

破れた相では1体分布が `P(s) ∝ exp(β/(2N)·R·cos2(s−φ))` に偏る（φ＝自発選択方向）。
この背景のもとでの設定間実効相関を平均場最低次で構成：
`C_eff = A + (β/2N)·B`、`A=cos2(θ_a−θ_b)/(2N)`（素・周期）、
`B=R²·cos2(θ_a−φ)·cos2(θ_b−φ)`（背景誘起）。

In [3]:
def solve_phi_R(beta, grid=4000, iters=6000):
    s = np.linspace(0,2*np.pi,grid,endpoint=False); c2,s2 = np.cos(2*s), np.sin(2*s)
    mx,my = 0.3, 0.0
    for _ in range(iters):
        f=(beta/(2*N))*(mx*c2+my*s2); f-=f.max(); w=np.exp(f); w/=w.sum()
        nx,ny=(w*c2).sum(),(w*s2).sum()
        if abs(nx-mx)+abs(ny-my)<1e-14: mx,my=nx,ny; break
        mx,my=nx,ny
    return np.hypot(mx,my), 0.5*np.arctan2(my,mx)

def analyze(beta, m=120):
    R,phi = solve_phi_R(beta)
    th = np.linspace(0,2*np.pi,m,endpoint=False)
    A = np.cos(2*(th[:,None]-th[None,:]))/(2*N)
    B = R*R*np.cos(2*(th[:,None]-phi))*np.cos(2*(th[None,:]-phi))
    C = A + (beta/(2*N))*B
    # 第1行のフーリエ成分（実効相関の角度構造）
    F = np.fft.rfft(C[0,:])/m; power = np.abs(F)**2
    topk = np.argsort(power)[::-1][:5]
    ev = np.sort(np.abs(eigh(0.5*(C+C.T))[0]))[::-1]
    rank_eff = int(np.sum(ev > 1e-9*ev[0]))
    return R, topk, rank_eff, ev[:6]

print("beta : R       支配フーリエモード k         rank(C_eff)   上位|λ|")
for beta in [8,12.5,14,16,20,30]:
    R, topk, rk, ev = analyze(beta)
    print(f"{beta:5.1f}: {R:.4f}  top-k={list(map(int,topk))}  rank={rk}")
    print(f"        |λ|: {np.round(ev,4)}")

beta : R       支配フーリエモード k         rank(C_eff)   上位|λ|
  8.0: 0.0000  top-k=[2, 1, 0, 42, 3]  rank=2
        |λ|: [10. 10.  0.  0.  0.  0.]
 12.5: 0.2791  top-k=[2, 1, 42, 38, 0]  rank=2
        |λ|: [19.7343 10.      0.      0.      0.      0.    ]
 14.0: 0.5088  top-k=[2, 42, 38, 1, 0]  rank=2
        |λ|: [46.2427 10.      0.      0.      0.      0.    ]
 16.0: 0.6472  top-k=[2, 1, 0, 42, 3]  rank=2
        |λ|: [77.0266 10.      0.      0.      0.      0.    ]
 20.0: 0.7731  top-k=[2, 1, 0, 42, 3]  rank=2
        |λ|: [129.5459  10.       0.       0.       0.       0.    ]
 30.0: 0.8768  top-k=[2, 1, 0, 42, 3]  rank=2
        |λ|: [240.6458  10.       0.       0.       0.       0.    ]


**判定II の結果**：破れた相でも実効相関は **rank 2**、支配モードは **k=2** のまま。
対称性の破れ（R>0）が起きても、実効相関の角度構造は周期的なまま。固有値は片方が `R²` で
伸びるが、これは「選ばれた周期モードの振幅増大」にすぎず、**新しい非周期モードは現れない**。

物理的には：破れたのは円上の方向（O(2) で φ を選んだ）であって、**周期性そのものは破れていない**。
選ばれたモード `cos2(θ−φ)` も k=2 の周期モード。`sign(cos2θ)` の π/4 ごとの周期反転は温存。

→ **判定II は（平均場最低次では）通らない**：相関長は発散の入口に立ったが、符号の問題は
解けていない。A-7 の状況（非コンパクトでも時間が単一にならない）が**作用面でも再現**された。

## 10.4 サマリー（established / open）

### established（この notebook で確定）
1. 全結合・符号つき `cos2` 結合の平均場は **連続相転移**を持ち、臨界点は **`β*=4N`**
   （N=3 で 12.00）。与件の `N` から出力され、手で入れた量でない。シード非依存・
   `β*`以下で無秩序相、で本物と確認。（**判定I＝非コンパクト化の入口はクリア**）
2. しかし破れた相の**実効相関は rank 2・k=2 の周期構造を保つ**。対称性の破れは円上の
   方向選択にとどまり、周期性そのものは破れない。`sign(cos2θ)` の周期反転は解消しない。
   （**判定II＝符号は平均場最低次では通らない**）
3. よって A-7 の状況（「非コンパクト化とローレンツ化は別操作、相関長が伸びても時間は単一に
   ならない」）が、素の二点相関だけでなく**作用の実効相関（平均場）でも再現**された。

### open（次へ）
1. **ゆらぎ（平均場を超える補正）で非周期モードが立つか。** 上の判定IIは平均場**最低次**の
   結果。符号を破る非周期構造は、もし存在するなら連結相関の高次（2体以上のゆらぎ）に隠れて
   いる可能性がある。「平均場では見えないがゆらぎで見える」は統計力学で普通の状況。
   → 有限サイズ全結合系の**直接モンテカルロ**で実効相関行列を測定し、臨界点 `β*=12` 近傍で
   非周期モードが立つかを見る。これが open 1 の最終判定。**notebook 11（Julia）で実施**。
2. 判定I がクリアした以上、相関長発散相での MDS 復元幾何を直接見る（円が壊れるか）。open 1
   と接続。

### 教訓（D への追記）
- 臨界点が「綺麗に」出た（`β*=4N`）が、それは判定I（非コンパクト化）のみで、判定II（符号）は
  別問題。**臨界点の存在に飛びついて「本丸が出た」と言ってはならない**（教訓D-4）。本丸が
  閉じるのは判定II まで含めてゆらぎで検証した後。